# IY008 – All Experiments Trace Overview (`NEW`)

Plots the mean ± SD TF localisation trace for every (experiment, group, channel) combination, for two datasets:
- `4_transformed_exp_time_series_NEW/` — post-media-switch, steady-state-extracted traces
- `5_FULL_transformed_exp_time_series_NEW/` — full traces (pre- and post-switch, no steady-state crop)

**TF identity** is resolved in priority order:
1. Experiment-specific OMERO metadata from `get_exp_summary` (acquisition files)
2. Lab strain database (`strain_tf_database.csv`)
3. `Unknown (group X)` as final fallback

**Excluded:** Experiment 18446 — not properly recorded, excluded from all analyses.

**Layout:** one figure per experiment — rows = groups, columns = GFP / mCherry channels.

In [1]:
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, "/home/ianyang/wela/src")
sys.path.insert(0, "/home/ianyang/alibylite/src")

IY008_DIR     = Path("/home/ianyang/stochastic_simulations/experiments/EXP-25-IY008")
SS_DATA_DIR   = IY008_DIR / "4_transformed_exp_time_series_NEW"       # post-switch, steady-state-extracted CSVs
FULL_DATA_DIR = IY008_DIR / "5_FULL_transformed_exp_time_series_NEW"  # full-length CSVs (no SS crop)
WELA_DIR      = IY008_DIR / "2_wela_data_analysis"  # raw wela TSV outputs, one subdir per OMID
META_COLS     = ["id", "group", "experiment"]  # non-time columns present in every time-series CSV

plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})
sns.set_theme(style="whitegrid", palette="colorblind")

palette   = sns.color_palette("colorblind")
GFP_COLOR = palette[2]   # index 2 = green
MC_COLOR  = palette[3]   # index 3 = reddish-orange


## 1. Build Condition Map from TSV Filenames

In [2]:
def condition_from_tsv_name(tsv_name: str) -> str:
    """Infer experimental condition from TSV filename keywords or IY026 mapping CSV."""
    n = tsv_name.lower()

    # Step 1: direct keyword matching for unambiguous filenames
    if "zero_glc" in n or "zeroglc" in n:
        return "0% glucose"
    if "lown2" in n or "low_n2" in n:
        return "low N₂"

    # Step 2: fall back to IY026 condition mapping CSV (contains OMID → condition assignments)
    mapping_csv = Path("/home/ianyang/stochastic_simulations/experiments/EXP-26-IY026") / "results.csv"
    mapping_df = pd.read_csv(mapping_csv)
    omid = tsv_name.split("_")[0]  # filename always starts with {omid}_
    mapped_condition = mapping_df[mapping_df["dataset_id"] == int(omid)]["condition"]

    if not mapped_condition.empty:
        # Normalise the IY026 label to match the naming used here
        if mapped_condition.values[0] == "2% to 0% glucose in SC":
            return "0% glucose"
        return mapped_condition.values[0]

    return "Unknown condition"


# Build condition map: scan each numeric OMID subdirectory for its dated TSV filename
CONDITION_MAP = {}
for omid_dir in sorted(WELA_DIR.iterdir()):
    if not omid_dir.is_dir() or not omid_dir.name.isdigit():
        continue
    # The dated TSV (e.g. "2801_2023_...tsv") is the primary data file; skip bare "{omid}.tsv"
    candidates = [p for p in omid_dir.glob(f"{omid_dir.name}_2*.tsv")]
    if candidates:
        CONDITION_MAP[omid_dir.name] = condition_from_tsv_name(candidates[0].name)

print(f"Condition map ({len(CONDITION_MAP)} OMIDs):")
for k, v in sorted(CONDITION_MAP.items()):
    print(f"  {k}: {v}")


Condition map (44 OMIDs):
  1613: Aim: Assess phototoxicity for 1min sampling rate
Strain: Msn4-GFP/Msn2-mCherry (901)
Comments:
OD of O/N: 1.75
OD after loading: 0.67
Was ~30 min late collecting log culture
Microscope setup for
  18360: Unknown condition
  18367: Unknown condition
  18446: Unknown condition
  18464: Unknown condition
  18572: Unknown condition
  18589: Unknown condition
  1908: Aim:   Strain:   Comments:   
Microscope setup for used channels:
Brightfield:
White LED
->(Polariser + Prism + condenser)]
->Filter block:[GFP/mCherry (59022) exciter,GFP/mCherry (59022) dichroi
  19316: Unknown condition
  19330: Unknown condition
  19391: Unknown condition
  19392: Unknown condition
  19394: Unknown condition
  19477: Unknown condition
  19554: Unknown condition
  19566: Unknown condition
  20213: Unknown condition
  2801: 0% glucose
  2841: 0% glucose
  2842: 0% glucose
  2843: 0% glucose
  2844: 0% glucose
  2849: 0% glucose
  2852: low N₂
  2853: low N₂
  2854: low N₂
  2

## 2. Helper Functions

In [ ]:
# File pattern: {numeric_omid}_group_{chN_TFNAME}_{GFP|mCherry}_time_series.csv
_FILE_RE = re.compile(r"^(\d+)_group_(.+?)_(GFP|mCherry)_time_series$")

# Manual overrides for group_ids that are strain IDs rather than encoded TF names.
# Keys must exactly match the group_id extracted from the filename (case-sensitive).
GROUP_TF_OVERRIDE = {
    "YST_1532":     "Sch9",
    # "Sse2_poss_mix": "Sse2",
    # "Gcd6_poss_mix": "Gcd6", # possibly a mix...
    # "Hsp82_prob_empty" : "Empty",
}


def get_tf(group_id: str) -> str:
    """Resolve TF name from group_id.

    Priority order:
      1. Manual override (GROUP_TF_OVERRIDE) — for strain-ID-style group_ids
      2. Strip leading 'chN_' prefix (e.g. 'ch10_YOX1' → 'YOX1')
      3. Return group_id as-is if no prefix matched
    """
    if group_id in GROUP_TF_OVERRIDE:
        return GROUP_TF_OVERRIDE[group_id]
    return re.sub(r"^ch\d+_", "", group_id)


def normalise_tf(name: str) -> str:
    """Normalise TF name to title case so that the same TF from different sources matches."""
    return name.strip().title()


def build_file_df(data_dir: Path) -> pd.DataFrame:
    """Scan data_dir for NEW time-series CSVs and return a metadata DataFrame."""
    records = []
    for csv_path in sorted(data_dir.glob("*.csv")):
        m = _FILE_RE.match(csv_path.stem)
        if m is None:
            print(f"  Skipping (unrecognised pattern): {csv_path.name}")
            continue
        exp_id, group_id, channel = m.group(1), m.group(2), m.group(3)
        records.append({
            "path":      csv_path,
            "exp_id":    exp_id,
            "group_id":  group_id,
            "channel":   channel,
            "condition": CONDITION_MAP.get(exp_id, "unknown"),
            "tf":        normalise_tf(get_tf(group_id)),
        })
    df = pd.DataFrame(records).sort_values(["exp_id", "group_id", "channel"])
    print(f"Found {len(df)} files across {df['exp_id'].nunique()} experiments in {data_dir.name}")
    return df


def plot_trace(ax, csv_path, color):
    """Load CSV, plot mean ± SD across cells. Returns (n_cells, n_timepoints)."""
    df = pd.read_csv(csv_path)
    # Identify time columns by excluding fixed metadata columns
    time_cols = [c for c in df.columns if c not in META_COLS]
    X    = df[time_cols].values.astype(float)
    t    = np.arange(X.shape[1])
    mean = np.nanmean(X, axis=0)
    sd   = np.nanstd(X,  axis=0, ddof=1)  # ddof=1 for sample SD
    ax.plot(t, mean, color=color, linewidth=1.8)
    ax.fill_between(t, mean - sd, mean + sd, color=color, alpha=0.2)
    ax.set_xlabel("Timepoint / 2 min")
    ax.set_ylabel("TF Localisation")
    ax.grid(alpha=0.3)
    return X.shape[0], X.shape[1]


def plot_all_experiments(file_df: pd.DataFrame, fig_prefix: str, IMG_DIR: Path):
    """One figure per experiment; rows = TF groups, cols = GFP / mCherry."""
    IMG_DIR.mkdir(parents=True, exist_ok=True)
    for exp_id in sorted(file_df["exp_id"].unique()):
        exp_data  = file_df[file_df["exp_id"] == exp_id]
        condition = CONDITION_MAP.get(exp_id, "unknown")
        groups    = sorted(exp_data["group_id"].unique())
        channels  = ["GFP", "mCherry"]

        n_rows = len(groups)
        n_cols = len(channels)
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(5 * n_cols, 3.5 * n_rows),
            constrained_layout=True,
        )
        # Ensure axes is always 2D regardless of n_rows / n_cols being 1
        axes = np.asarray(axes).reshape(n_rows, n_cols)

        for row, group_id in enumerate(groups):
            # Read the already-normalised TF name from file_df rather than re-deriving it
            tf_name = exp_data[exp_data["group_id"] == group_id]["tf"].iloc[0]
            for col, channel in enumerate(channels):
                ax    = axes[row, col]
                match = exp_data[
                    (exp_data["group_id"] == group_id) &
                    (exp_data["channel"]  == channel)
                ]
                if match.empty:
                    ax.set_visible(False)
                    continue
                color = GFP_COLOR if channel == "GFP" else MC_COLOR
                n_cells, n_tp = plot_trace(ax, match.iloc[0]["path"], color)
                ax.set_title(f"{tf_name} ({channel})\nn = {n_cells} cells, {n_tp} tp")

        fig.suptitle(
            f"Experiment {exp_id} — {condition}\n(shaded band = mean ± SD)",
            fontsize=14,
        )
        fig_path = IMG_DIR / f"{fig_prefix}_exp{exp_id}.png"
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {fig_path}")
        # plt.show()


## 3. Post-Media-Switch & Steady-State Traces (`4_transformed_exp_time_series_NEW`)

In [4]:
ss_file_df = build_file_df(SS_DATA_DIR)
display(ss_file_df[["exp_id", "group_id", "channel", "condition", "tf"]].reset_index(drop=True))

Found 601 files across 25 experiments in 4_transformed_exp_time_series_NEW


,exp_id,group_id,channel,condition,tf
0,2801,ch10_YOX1,GFP,0% glucose,Yox1
1,2801,ch10_YOX1,mCherry,0% glucose,Yox1
2,2801,ch11_CUP9,GFP,0% glucose,Cup9
3,2801,ch11_CUP9,mCherry,0% glucose,Cup9
4,2801,ch12_CIN5,GFP,0% glucose,Cin5
...,...,...,...,...,...
596,4251,Swi6,GFP,low N₂,Swi6
597,4251,Ume6,GFP,low N₂,Ume6
598,4251,Upc2,GFP,low N₂,Upc2
599,4251,Yrr1,GFP,low N₂,Yrr1


In [ ]:
plot_all_experiments(ss_file_df, fig_prefix="IY008_NEW_trace_overview",
                     IMG_DIR=SS_DATA_DIR / "trace_overviews")

## 4. Full Traces (`5_FULL_transformed_exp_time_series_NEW`)

In [6]:
full_file_df = build_file_df(FULL_DATA_DIR)
display(full_file_df[["exp_id", "group_id", "channel", "condition", "tf"]].reset_index(drop=True))
# save metadata to CSV for reference
full_file_df[["exp_id", "group_id", "channel", "condition", "tf"]].reset_index(drop=True).to_csv(IY008_DIR / "NEW_data_metadata.csv", index=False)


Found 639 files across 26 experiments in 5_FULL_transformed_exp_time_series_NEW


,exp_id,group_id,channel,condition,tf
0,2801,ch10_YOX1,GFP,0% glucose,Yox1
1,2801,ch10_YOX1,mCherry,0% glucose,Yox1
2,2801,ch11_CUP9,GFP,0% glucose,Cup9
3,2801,ch11_CUP9,mCherry,0% glucose,Cup9
4,2801,ch12_CIN5,GFP,0% glucose,Cin5
...,...,...,...,...,...
634,4251,Swi6,GFP,low N₂,Swi6
635,4251,Ume6,GFP,low N₂,Ume6
636,4251,Upc2,GFP,low N₂,Upc2
637,4251,Yrr1,GFP,low N₂,Yrr1


In [ ]:
plot_all_experiments(full_file_df, fig_prefix="IY008_FULL_NEW_trace_overview",
                     IMG_DIR=FULL_DATA_DIR / "trace_overviews")


## 5. Per-TF Traces Across Conditions

One figure per unique TF — mean ± SD traces coloured by condition, pooling all channels and experiments that track that TF.

In [8]:
def plot_tf_across_conditions(file_df: pd.DataFrame, fig_prefix: str, IMG_DIR: Path):
    """One figure per unique TF: mean ± SD per condition, pooling all channels and experiments.

    Multiple group_ids resolving to the same TF (e.g. ch10_YOX1 measured in GFP and mCherry)
    are pooled together; n reported is cell-channel pairs, not unique physical cells.
    This is intentional — both channels track the same TF signal.
    """
    IMG_DIR.mkdir(parents=True, exist_ok=True)
    all_tfs    = sorted(file_df["tf"].unique())
    conditions = sorted(file_df["condition"].unique())
    cond_palette = sns.color_palette("colorblind", n_colors=max(len(conditions), 1))
    cond_colors  = {cond: cond_palette[i] for i, cond in enumerate(conditions)}

    for tf in all_tfs:
        tf_rows  = file_df[file_df["tf"] == tf]
        fig, ax  = plt.subplots(figsize=(10, 4), constrained_layout=True)
        plotted  = False

        for cond in conditions:
            cond_rows = tf_rows[tf_rows["condition"] == cond]
            if cond_rows.empty:
                continue

            # Collect all cell matrices for this TF + condition (may span multiple channels/experiments)
            arrays = []
            for _, row in cond_rows.iterrows():
                df = pd.read_csv(row["path"])
                time_cols = [c for c in df.columns if c not in META_COLS]
                arrays.append(df[time_cols].values.astype(float))

            # Trim to the shortest series before stacking so array shapes align
            min_tp = min(a.shape[1] for a in arrays)
            X_pool = np.vstack([a[:, :min_tp] for a in arrays])

            t     = np.arange(min_tp)
            mean  = np.nanmean(X_pool, axis=0)
            sd    = np.nanstd(X_pool,  axis=0, ddof=1)
            color = cond_colors[cond]

            # n = total cell-channel pairs pooled (not unique physical cells)
            ax.plot(t, mean, label=f"{cond}  (n={X_pool.shape[0]} cell-channel pairs)", color=color, linewidth=1.8)
            ax.fill_between(t, mean - sd, mean + sd, color=color, alpha=0.15)
            plotted = True

        if not plotted:
            plt.close(fig)
            continue

        ax.set_xlabel("Timepoint / 2 min")
        ax.set_ylabel("TF Localisation")
        ax.grid(alpha=0.3)
        ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False, title="Condition")
        fig.suptitle(f"{tf} — mean ± SD across conditions", fontsize=14)
        fig_path = IMG_DIR / f"{fig_prefix}_{tf}.png"
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {fig_path}")
        # plt.show()


### Steady-state traces (`4_transformed_exp_time_series_NEW`)

In [ ]:
plot_tf_across_conditions(ss_file_df, fig_prefix="IY008_NEW_TF_conditions",
                          IMG_DIR=SS_DATA_DIR / "TF_conditions_overviews")


### Full traces (`5_FULL_transformed_exp_time_series_NEW`)

In [ ]:
plot_tf_across_conditions(full_file_df, fig_prefix="IY008_FULL_NEW_TF_conditions",
                          IMG_DIR=FULL_DATA_DIR / "TF_conditions_overviews")